# Stage 01 — 2.5D Cache Streaming
**Medical-Grade Kaggle Edition — Uses Approved Stage 00 Manifest**

Mục tiêu:
- Đọc `q1_clean_manifest_for_cache.csv` từ output Stage 00 đã được PM duyệt.
- Không full-extract toàn bộ dataset.
- Stream extract chỉ CT/PET `.npy` cần dùng theo từng archive/source_part.
- Tạo 2.5D cache dạng `np.memmap` uint8, shape `(N, 3, 224, 224)`:
  - channel 0: CT central axial slice
  - channel 1: PET maximum intensity projection
  - channel 2: fused overlay proxy
- Xuất metadata cho Stage 03.

> PM Gate: Output Stage 01 phải được review trước khi chạy Stage 03.

In [ ]:
# CELL 1 — Kaggle-only 7z install/check
try:
    get_ipython  # noqa: F821
    get_ipython().system("which 7z || which 7zz || which 7za || echo 'WARNING: 7z/7zz/7za not found; enable Kaggle image with 7z or install p7zip before running Stage 01.'")
except NameError:
    pass

In [ ]:
# CELL 2 — Imports + Configuration
from pathlib import Path
from collections import defaultdict
import os
import re
import json
import shutil
import hashlib
import subprocess
import warnings
import gc
import shutil as _shutil_disk

import numpy as np
import pandas as pd
from IPython.display import display

try:
    from PIL import Image
except Exception as exc:
    raise RuntimeError("Stage 01 cần Pillow/PIL để resize ảnh 2.5D.") from exc

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 220)

IS_KAGGLE = Path("/kaggle").exists()
OUTPUT_ROOT = Path("/kaggle/working/vimedpet_q1_outputs") if IS_KAGGLE else Path("vimedpet_q1_outputs")
STAGE00_DIR = OUTPUT_ROOT / "00_manifest"
CACHE_DIR = OUTPUT_ROOT / "01_cache"
TABLE_DIR = CACHE_DIR / "tables"
FIGURE_DIR = CACHE_DIR / "figures"
SCRATCH_ROOT = Path(os.environ.get(
    "STAGE01_SCRATCH_ROOT",
    str((OUTPUT_ROOT / "_scratch_stage01") if IS_KAGGLE else (OUTPUT_ROOT / "_scratch_stage01")),
))
STAGE_ROOT = SCRATCH_ROOT / "archive_stage"
EXTRACT_ROOT = SCRATCH_ROOT / "extracted_npy"

for p in [CACHE_DIR, TABLE_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)
if SCRATCH_ROOT.exists():
    shutil.rmtree(SCRATCH_ROOT, ignore_errors=True)
for p in [SCRATCH_ROOT, STAGE_ROOT, EXTRACT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 224
CACHE_SHAPE_CHANNELS = 3
CACHE_DTYPE = "uint8"
# Giữ batch nhỏ để tránh đầy /tmp/system disk trên Kaggle.
# Có thể tăng bằng env `STAGE01_BATCH_EXTRACT_SIZE` nếu disk còn dư.
BATCH_EXTRACT_SIZE = int(os.environ.get("STAGE01_BATCH_EXTRACT_SIZE", "8"))
SMOKE_TEST_N = int(os.environ.get("STAGE01_SMOKE_TEST_N", "0"))
RECOMMENDED_MAX_SEQ_LENGTH = 512

print("IS_KAGGLE:", IS_KAGGLE)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("CACHE_DIR:", CACHE_DIR)
print("SCRATCH_ROOT:", SCRATCH_ROOT)
print("BATCH_EXTRACT_SIZE:", BATCH_EXTRACT_SIZE)
print("SMOKE_TEST_N:", SMOKE_TEST_N)


def disk_report(path: Path, label: str):
    """In dung lượng disk để bắt sớm lỗi high /tmp/system disk trên Kaggle."""
    try:
        usage = _shutil_disk.disk_usage(path)
    except Exception:
        usage = _shutil_disk.disk_usage(path.anchor or "/")
    gb = 1024 ** 3
    print(
        f"[disk:{label}] path={path} "
        f"total={usage.total / gb:.2f}GB used={usage.used / gb:.2f}GB free={usage.free / gb:.2f}GB"
    )
    return usage


def assert_min_free_space(path: Path, min_gb: float, label: str):
    usage = disk_report(path, label)
    free_gb = usage.free / (1024 ** 3)
    if free_gb < min_gb:
        raise RuntimeError(
            f"Không đủ disk trống cho Stage 01 tại {path}: free={free_gb:.2f}GB < {min_gb:.2f}GB. "
            "Hãy restart Kaggle session, xóa output cũ, hoặc giảm STAGE01_BATCH_EXTRACT_SIZE."
        )

assert_min_free_space(OUTPUT_ROOT, 2.0, "startup-output")
assert_min_free_space(SCRATCH_ROOT, 2.0, "startup-scratch")

In [ ]:
# CELL 3 — Locate Stage 00 Manifest
STAGE00_INPUT_HINTS = [
    # Output khi chạy notebook Stage 00 trong cùng session.
    Path("/kaggle/working/vimedpet_q1_outputs/00_manifest"),
    # User-provided Kaggle notebook input path.
    Path("/kaggle/input/notebooks/tundng111/00-archive-listing-manifest-clean"),
    Path("/kaggle/input/notebooks/tundng111/00-archive-listing-manifest-clean/vimedpet_q1_outputs/00_manifest"),
    Path("/kaggle/input/notebooks/tundng111/00-archive-listing-manifest-clean/00_manifest"),
    # Common Kaggle mounted notebook output variants.
    Path("/kaggle/input/00-archive-listing-manifest-clean"),
    Path("/kaggle/input/00-archive-listing-manifest-clean/vimedpet_q1_outputs/00_manifest"),
    Path("/kaggle/input/00-archive-listing-manifest-clean/00_manifest"),
    # Local/dev fallback.
    STAGE00_DIR,
]


def find_stage00_manifest() -> Path:
    candidates = []
    for root in STAGE00_INPUT_HINTS:
        candidates.append(root / "q1_clean_manifest_for_cache.csv")
        candidates.append(root / "00_manifest" / "q1_clean_manifest_for_cache.csv")
        candidates.append(root / "vimedpet_q1_outputs" / "00_manifest" / "q1_clean_manifest_for_cache.csv")
    # Fallback bounded scan under /kaggle/input for notebook output datasets.
    if Path("/kaggle/input").exists():
        candidates.extend(Path("/kaggle/input").rglob("q1_clean_manifest_for_cache.csv"))
    seen = set()
    uniq = []
    for p in candidates:
        try:
            rp = p.resolve()
        except Exception:
            rp = p
        if rp not in seen:
            uniq.append(p)
            seen.add(rp)
    print("Manifest candidates checked:")
    for p in uniq[:80]:
        print(" -", p, "exists=", p.exists())
    for p in uniq:
        if p.exists() and p.is_file():
            return p
    raise FileNotFoundError(
        "Không tìm thấy q1_clean_manifest_for_cache.csv. "
        "Hãy Add Data output Stage 00 notebook vào Kaggle input."
    )

MANIFEST_PATH = find_stage00_manifest()
STAGE00_RESOLVED_DIR = MANIFEST_PATH.parent
print("Using Stage 00 manifest:", MANIFEST_PATH)
print("Stage 00 dir:", STAGE00_RESOLVED_DIR)

manifest = pd.read_csv(MANIFEST_PATH)
if SMOKE_TEST_N > 0:
    manifest = manifest.head(SMOKE_TEST_N).copy()
required_cols = [
    "case_id", "source_part", "patient_key", "ct_path", "pet_path",
    "report_path", "report_text_clean", "clean_split", "row_index",
]
missing = [c for c in required_cols if c not in manifest.columns]
assert not missing, f"Stage 00 manifest thiếu cột bắt buộc: {missing}"
assert len(manifest) > 0, "Manifest Stage 00 rỗng."
assert manifest["case_id"].duplicated().sum() == 0, "Duplicate case_id trong manifest."
assert manifest["row_index"].duplicated().sum() == 0, "Duplicate row_index trong manifest."
print("Manifest rows:", len(manifest))
print("Split counts:")
print(manifest["clean_split"].value_counts().to_string())
display(manifest.head())

In [ ]:
# CELL 4 — Discover raw archive markers
RAW_INPUT_HINTS = [
    Path("/kaggle/input/vimed-pet-ct-part1-raw-no-autoextract-v3"),
    Path("/kaggle/input/vimed-pet-ct-part2-raw-no-autoextract-v1"),
    Path("/kaggle/input/vimed-pet-ct-part3-raw-no-autoextract-v1"),
    Path("/kaggle/input/datasets/tundng111/vimed-pet-ct-part1-raw-no-autoextract-v3"),
    Path("/kaggle/input/datasets/tundng111/vimed-pet-ct-part2-raw-no-autoextract-v1"),
    Path("/kaggle/input/datasets/tundng111/vimed-pet-ct-part3-raw-no-autoextract-v1"),
    Path("/kaggle/input"),
    Path.cwd(),
]


def existing_roots():
    roots, seen = [], set()
    for root in RAW_INPUT_HINTS:
        if root.exists():
            try:
                rr = root.resolve()
            except Exception:
                rr = root
            if rr not in seen:
                roots.append(rr)
                seen.add(rr)
    return roots


def archive_stem_from_marker(marker: Path) -> str:
    name = marker.name
    if name.endswith(".zipraw"):
        return name[:-len(".zipraw")]
    if name.endswith(".zip"):
        return name[:-len(".zip")]
    return marker.stem

roots = existing_roots()
print("Raw roots:")
for r in roots:
    print(" -", r)

all_markers = []
for root in roots:
    all_markers.extend(root.rglob("*.zipraw"))
    all_markers.extend(root.rglob("*.zip"))
all_markers = sorted(set(all_markers))

archive_groups = {}
for marker in all_markers:
    stem = archive_stem_from_marker(marker)
    if stem in archive_groups and marker.suffix != ".zipraw":
        continue
    parts = sorted(marker.parent.glob(f"{stem}.z[0-9][0-9]"))
    archive_groups[stem] = {
        "stem": stem,
        "folder": marker.parent,
        "marker": marker,
        "parts": parts,
    }

needed_stems = sorted(manifest["source_part"].astype(str).unique().tolist())
print("Needed source_part/archive stems:", needed_stems)
print("Discovered archive groups:", sorted(archive_groups))
missing_stems = [s for s in needed_stems if s not in archive_groups]
assert not missing_stems, f"Không tìm thấy raw archive cho source_part: {missing_stems}"
for stem in needed_stems:
    g = archive_groups[stem]
    print(f"- {stem}: marker={g['marker']}, parts={len(g['parts'])}")

In [ ]:
# CELL 5 — 7z helpers

def sevenzip_binary():
    for exe in ["7z", "7zz", "7za"]:
        try:
            subprocess.run([exe], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            return exe
        except Exception:
            pass
    raise RuntimeError("Không tìm thấy 7z/7zz/7za.")

SEVENZIP = sevenzip_binary()
print("Using:", SEVENZIP)


def safe_unlink(path: Path):
    if path.exists() or path.is_symlink():
        path.unlink()


def symlink_required(src: Path, dst: Path):
    """Symlink archive split parts; không copy file lớn để tránh đầy Kaggle disk."""
    safe_unlink(dst)
    try:
        os.symlink(src, dst)
    except Exception as exc:
        raise RuntimeError(
            f"Không tạo được symlink từ {src} -> {dst}. "
            "Không fallback copy vì archive rất lớn và có thể làm đầy disk Kaggle."
        ) from exc


def prepare_archive_stage(stem: str) -> Path:
    g = archive_groups[stem]
    stage = STAGE_ROOT / stem
    if stage.exists():
        shutil.rmtree(stage, ignore_errors=True)
    stage.mkdir(parents=True, exist_ok=True)
    for p in g["parts"]:
        symlink_required(p, stage / p.name)
    zip_path = stage / f"{stem}.zip"
    symlink_required(g["marker"], zip_path)
    return zip_path


def cleanup_archive_stage(stem: str):
    stage = STAGE_ROOT / stem
    if stage.exists():
        shutil.rmtree(stage, ignore_errors=True)


def run_7z_extract_specific(zip_path: Path, out_dir: Path, inner_paths):
    inner_paths = [str(p) for p in inner_paths if str(p).strip()]
    if not inner_paths:
        return
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [SEVENZIP, "x", str(zip_path), f"-o{out_dir}", "-y"] + inner_paths
    r = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if r.returncode != 0:
        raise RuntimeError(
            "7z extract failed\n"
            f"CMD prefix: {' '.join(cmd[:6])} ... n_files={len(inner_paths)}\n"
            f"STDOUT:\n{r.stdout[-2000:]}\n"
            f"STDERR:\n{r.stderr[-4000:]}"
        )

In [ ]:
# CELL 6 — Image helpers

def to_numpy_safe(x) -> np.ndarray:
    arr = np.asarray(x)
    if arr.ndim == 4:
        # Nếu có channel dimension singleton, squeeze an toàn.
        arr = np.squeeze(arr)
    return arr


def ensure_3d_volume(arr: np.ndarray, name: str) -> np.ndarray:
    arr = to_numpy_safe(arr)
    if arr.ndim == 2:
        arr = arr[None, :, :]
    if arr.ndim != 3:
        raise ValueError(f"{name} expected 2D/3D after squeeze, got shape={arr.shape}")
    return arr.astype(np.float32, copy=False)


def robust_norm(x: np.ndarray, lo=1, hi=99) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    finite = np.isfinite(x)
    if not finite.any():
        return np.zeros_like(x, dtype=np.float32)
    vals = x[finite]
    a, b = np.percentile(vals, [lo, hi])
    if not np.isfinite(a) or not np.isfinite(b) or b <= a:
        return np.zeros_like(x, dtype=np.float32)
    y = np.clip((x - a) / (b - a), 0, 1)
    y[~finite] = 0
    return y.astype(np.float32, copy=False)


def resize_uint8(img01: np.ndarray, size=IMAGE_SIZE) -> np.ndarray:
    img01 = np.clip(img01, 0, 1)
    pil = Image.fromarray((img01 * 255).astype(np.uint8))
    pil = pil.resize((size, size), resample=Image.BILINEAR)
    return np.asarray(pil, dtype=np.uint8)


def build_25d_from_arrays(ct_arr: np.ndarray, pet_arr: np.ndarray) -> np.ndarray:
    ct = ensure_3d_volume(ct_arr, "ct")
    pet = ensure_3d_volume(pet_arr, "pet")
    ct_slice = robust_norm(ct[ct.shape[0] // 2], 1, 99)
    pet_mip = robust_norm(pet, 1, 99.5).max(axis=0)
    ct_u8 = resize_uint8(ct_slice)
    pet_u8 = resize_uint8(pet_mip)
    overlay01 = np.clip(0.65 * (ct_u8.astype(np.float32) / 255.0) + 0.35 * (pet_u8.astype(np.float32) / 255.0), 0, 1)
    overlay_u8 = (overlay01 * 255).astype(np.uint8)
    return np.stack([ct_u8, pet_u8, overlay_u8], axis=0)


def stable_error_id(*items) -> str:
    key = "|".join(map(str, items)).encode("utf-8")
    return hashlib.sha1(key).hexdigest()[:12]

In [ ]:
# CELL 7 — Build memmap cache by streaming archives
N = len(manifest)
memmap_path = CACHE_DIR / f"q1_25d_uint8_{IMAGE_SIZE}.memmap"
cache_index_path = CACHE_DIR / "q1_cache_index.csv"
cache_manifest_path = CACHE_DIR / "q1_clean_manifest_for_cache.csv"
error_path = CACHE_DIR / "q1_cache_errors.csv"

if memmap_path.exists():
    memmap_path.unlink()
cache = np.memmap(memmap_path, dtype=np.uint8, mode="w+", shape=(N, CACHE_SHAPE_CHANNELS, IMAGE_SIZE, IMAGE_SIZE))
cache[:] = 0
cache.flush()

index_rows = []
error_rows = []
processed = 0

for stem in needed_stems:
    part_df = manifest[manifest["source_part"].astype(str) == stem].copy()
    part_df = part_df.sort_values("row_index").reset_index(drop=True)
    print("\n" + "=" * 80)
    print(f"[Stage01] Processing {stem}: rows={len(part_df)}")
    disk_report(OUTPUT_ROOT, f"before-{stem}-output")
    disk_report(SCRATCH_ROOT, f"before-{stem}-scratch")
    zip_path = prepare_archive_stage(stem)
    extract_dir = EXTRACT_ROOT / stem
    if extract_dir.exists():
        shutil.rmtree(extract_dir, ignore_errors=True)
    extract_dir.mkdir(parents=True, exist_ok=True)
    try:
        # Disk-safe streaming: extract một micro-batch, xử lý, rồi xóa ngay.
        # Tuyệt đối không extract toàn bộ source_part cùng lúc vì dễ đầy /tmp/system disk Kaggle.
        for start in range(0, len(part_df), BATCH_EXTRACT_SIZE):
            batch_df = part_df.iloc[start:start + BATCH_EXTRACT_SIZE].copy()
            batch_no = start // BATCH_EXTRACT_SIZE + 1
            needed_paths = []
            for _, row in batch_df.iterrows():
                needed_paths.append(str(row["ct_path"]))
                needed_paths.append(str(row["pet_path"]))
            needed_paths = sorted(set(needed_paths))
            print(f"  batch {batch_no}: rows={len(batch_df)}, extract_npy={len(needed_paths)}")
            assert_min_free_space(SCRATCH_ROOT, 1.0, f"pre-extract-{stem}-batch{batch_no}")

            # Dọn folder extract trước mỗi batch để không tích lũy file lớn.
            shutil.rmtree(extract_dir, ignore_errors=True)
            extract_dir.mkdir(parents=True, exist_ok=True)
            run_7z_extract_specific(zip_path, extract_dir, needed_paths)

            for local_i, (_, row) in enumerate(batch_df.iterrows(), start=start + 1):
                global_i = int(row["row_index"])
                ct_file = extract_dir / str(row["ct_path"])
                pet_file = extract_dir / str(row["pet_path"])
                status = "ok"
                err = ""
                try:
                    if not ct_file.exists():
                        raise FileNotFoundError(f"missing CT file after extract: {ct_file}")
                    if not pet_file.exists():
                        raise FileNotFoundError(f"missing PET file after extract: {pet_file}")
                    ct_arr = np.load(ct_file, mmap_mode="r")
                    pet_arr = np.load(pet_file, mmap_mode="r")
                    img = build_25d_from_arrays(ct_arr, pet_arr)
                    cache[global_i] = img
                    ct_shape = tuple(np.asarray(ct_arr).shape)
                    pet_shape = tuple(np.asarray(pet_arr).shape)
                    del ct_arr, pet_arr, img
                except Exception as exc:
                    status = "error"
                    err = str(exc)
                    ct_shape = ""
                    pet_shape = ""
                    error_rows.append({
                        "case_id": row["case_id"],
                        "row_index": global_i,
                        "source_part": stem,
                        "error_id": stable_error_id(row["case_id"], err),
                        "error": err,
                        "ct_path": row["ct_path"],
                        "pet_path": row["pet_path"],
                    })
                index_rows.append({
                    "case_id": row["case_id"],
                    "row_index": global_i,
                    "source_part": stem,
                    "patient_key": row["patient_key"],
                    "clean_split": row["clean_split"],
                    "cache_path": str(memmap_path),
                    "cache_offset": global_i,
                    "shape": f"{CACHE_SHAPE_CHANNELS},{IMAGE_SIZE},{IMAGE_SIZE}",
                    "dtype": CACHE_DTYPE,
                    "status": status,
                    "error": err,
                    "ct_path": row["ct_path"],
                    "pet_path": row["pet_path"],
                    "report_path": row["report_path"],
                    "ct_shape": str(ct_shape),
                    "pet_shape": str(pet_shape),
                })
                processed += 1

            cache.flush()
            shutil.rmtree(extract_dir, ignore_errors=True)
            extract_dir.mkdir(parents=True, exist_ok=True)
            gc.collect()
            if batch_no == 1 or batch_no % 10 == 0 or start + len(batch_df) >= len(part_df):
                disk_report(SCRATCH_ROOT, f"post-clean-{stem}-batch{batch_no}")
            print(f"  processed {min(start + len(batch_df), len(part_df))}/{len(part_df)} in {stem}; total={processed}/{N}")
    finally:
        cleanup_archive_stage(stem)
        shutil.rmtree(extract_dir, ignore_errors=True)
        gc.collect()

cache.flush()
error_columns = ["case_id", "row_index", "source_part", "error_id", "error", "ct_path", "pet_path"]
index_df = pd.DataFrame(index_rows).sort_values("row_index").reset_index(drop=True)
error_df = pd.DataFrame(error_rows, columns=error_columns)
index_df.to_csv(cache_index_path, index=False, encoding="utf-8-sig")
error_df.to_csv(error_path, index=False, encoding="utf-8-sig")
manifest.to_csv(cache_manifest_path, index=False, encoding="utf-8-sig")
print("Cache build done.")
print("Rows:", len(index_df), "Errors:", len(error_df))
display(index_df.head())
if len(error_df):
    display(error_df.head())

In [ ]:
# CELL 8 — Validate cache and export metadata
index_df = pd.read_csv(cache_index_path)
error_df = pd.read_csv(error_path) if error_path.exists() and error_path.stat().st_size > 0 else pd.DataFrame()
status_counts = index_df["status"].value_counts().to_dict()
ok_rows = int((index_df["status"] == "ok").sum())
error_rows_n = int((index_df["status"] != "ok").sum())

# Re-open read-only to verify shape and sample non-empty values.
cache_ro = np.memmap(memmap_path, dtype=np.uint8, mode="r", shape=(N, CACHE_SHAPE_CHANNELS, IMAGE_SIZE, IMAGE_SIZE))
sample_indices = index_df.loc[index_df["status"] == "ok", "row_index"].head(8).astype(int).tolist()
sample_stats = []
for idx in sample_indices:
    arr = np.asarray(cache_ro[idx])
    sample_stats.append({
        "row_index": idx,
        "min": int(arr.min()),
        "max": int(arr.max()),
        "mean": float(arr.mean()),
        "nonzero_percent": float((arr > 0).mean() * 100),
    })
sample_stats_df = pd.DataFrame(sample_stats)
sample_stats_df.to_csv(TABLE_DIR / "01_cache_sample_stats.csv", index=False, encoding="utf-8-sig")
display(sample_stats_df)

split_counts = index_df.groupby(["clean_split", "status"]).size().reset_index(name="n")
split_counts.to_csv(TABLE_DIR / "02_cache_split_status.csv", index=False, encoding="utf-8-sig")
display(split_counts)

meta = {
    "stage": "01_build_25d_cache_streaming",
    "stage00_manifest_path": str(MANIFEST_PATH),
    "cache_path": str(memmap_path),
    "cache_index_path": str(cache_index_path),
    "cache_manifest_path": str(cache_manifest_path),
    "cache_error_path": str(error_path),
    "n_cases_manifest": int(N),
    "n_cache_rows": int(len(index_df)),
    "ok_rows": ok_rows,
    "error_rows": error_rows_n,
    "status_counts": {str(k): int(v) for k, v in status_counts.items()},
    "shape": [int(N), CACHE_SHAPE_CHANNELS, IMAGE_SIZE, IMAGE_SIZE],
    "dtype": CACHE_DTYPE,
    "image_size": IMAGE_SIZE,
    "channels": ["ct_central_slice", "pet_mip", "ct_pet_overlay_proxy"],
    "normalization": {
        "ct": "per-volume robust percentile 1-99 on central slice",
        "pet": "per-volume robust percentile 1-99.5 then axial MIP",
        "overlay": "0.65*CT + 0.35*PET after uint8 resize",
    },
    "split_counts": {str(k): int(v) for k, v in manifest["clean_split"].value_counts().to_dict().items()},
    "recommended_max_seq_length_for_stage03": RECOMMENDED_MAX_SEQ_LENGTH,
    "pm_gate": "PASS" if error_rows_n == 0 and ok_rows == N else "REVIEW_REQUIRED",
}
meta_path = CACHE_DIR / "q1_cache_meta.json"
meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(meta, ensure_ascii=False, indent=2))

assert len(index_df) == N, "Cache index row count không khớp manifest."
assert index_df["row_index"].duplicated().sum() == 0, "Duplicate row_index trong cache index."
assert ok_rows + error_rows_n == N, "Status count không khớp tổng số case."
if error_rows_n > 0:
    print("WARNING: Có lỗi cache rows. Cần PM review q1_cache_errors.csv trước Stage 03.")
else:
    print("STAGE 01 CACHE VALIDATION PASSED — no cache errors.")

In [ ]:
# CELL 9 — PM checkpoint
print("===== STAGE 01 DONE =====")
print("Output files:")
for p in sorted(CACHE_DIR.rglob("*")):
    if p.is_file():
        print(p, p.stat().st_size, "bytes")
print("\nPM CHECKPOINT:")
print("- Gửi q1_cache_meta.json để review trước Stage 03.")
print("- Không chạy Stage 03 nếu pm_gate != PASS hoặc q1_cache_errors.csv có lỗi.")